# Transfermarkt modeling starter
Run `python scripts/prepare_transfermarkt.py` from the project folder first.
This notebook reads the generated SQLite database. The example predicts whether an attacker
records at least **5 goals in the next season in the same league**, using the observed season's features.
This is a demonstration outcome, not a universal measure of player quality.


In [1]:
from pathlib import Path
import json
import sqlite3
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/processed/transfermarkt/transfermarkt.sqlite").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project after building Transfermarkt tables.")
DATA = ROOT / "data/processed/transfermarkt"
print(json.loads((DATA / "build_metadata.json").read_text())["tables"])
pd.read_csv(DATA / "coverage.csv").head()


{'player_match': 726823, 'player_season': 37419, 'player_club_season': 38555, 'team_match': 50482, 'matches': 25241, 'players': 50149, 'clubs': 796, 'competitions': 65, 'lineups': 929583, 'player_valuations': 237848}


,competition_id,league,season_start_year,games_with_appearances,players,appearances,missing_historical_values
0,ES1,La Liga,2012,380,540,10553,70
1,ES1,La Liga,2013,380,539,10586,68
2,ES1,La Liga,2014,380,468,9451,95
3,ES1,La Liga,2015,380,538,10521,95
4,ES1,La Liga,2016,380,541,10559,40


In [2]:
with sqlite3.connect(DATA / "transfermarkt.sqlite") as con:
    seasons = pd.read_sql_query("SELECT * FROM player_season", con)
    example_matches = pd.read_sql_query(
        "SELECT * FROM player_match WHERE competition_id = ? AND season_start_year = ? LIMIT 10",
        con, params=("GB1", 2023), parse_dates=["date"])
print(seasons.shape, example_matches.shape)
seasons.head()


(37419, 35) (10, 47)


,player_id,competition_id,league,season_start_year,season,player_name,position,sub_position,date_of_birth,height_cm,...,assists_per_match,goals_per90,assists_per90,market_value_before_first_match_date,market_value_before_first_match_eur,market_value_before_first_match_age_days,target_next_season_goals_per_season,target_next_season_assists_per_season,target_next_season_minutes_played,target_next_season_appearances
0,5592,FR1,Ligue 1,2012,2012-2013,Cyril Jeunechamp,Midfield,Defensive Midfield,1975-12-18,175.0,...,0.000000,0.000000,0.000000,2012-05-23,500000.0,79.0,NaN,NaN,NaN,NaN
1,126767,FR1,Ligue 1,2012,2012-2013,Ali Ahamada,Goalkeeper,Goalkeeper,1991-08-19,189.0,...,0.000000,0.028544,0.000000,2012-05-23,4000000.0,79.0,0.0,0.0,1890.0,21.0
2,5672,FR1,Ligue 1,2012,2012-2013,Étienne Didot,Midfield,Central Midfield,1983-07-24,175.0,...,0.250000,0.061856,0.278351,2012-05-23,4000000.0,79.0,1.0,2.0,2460.0,33.0
3,113707,FR1,Ligue 1,2012,2012-2013,Rémy Cabella,Midfield,Attacking Midfield,1990-03-08,172.0,...,0.225806,0.271084,0.271084,2012-05-23,3000000.0,79.0,14.0,7.0,3206.0,37.0
4,5495,FR1,Ligue 1,2012,2012-2013,John Utaka,Attack,Left Winger,1982-01-08,182.0,...,0.214286,0.250580,0.250580,2012-05-23,4000000.0,79.0,NaN,NaN,NaN,NaN


## Choose the cohort and hold out a season
Keep attackers with at least 450 observed minutes and an observed following-season outcome.
Missing next-season records are excluded, not treated as zero goals. This creates selection bias:
players who leave the league or disappear from the source are not evaluated.
The 2023 predictor season is the held-out test set (target: 2024). Earlier seasons train the model.
Use other earlier seasons for tuning rather than repeatedly optimizing against this test set.


In [3]:
TEST_YEAR = 2023
TARGET_GOALS = 5
observed_target = "target_next_season_goals_per_season"
cohort = seasons.loc[(seasons["position"] == "Attack") &
                     (seasons["minutes_played"] >= 450) & seasons[observed_target].notna()].copy()
cohort["target"] = (cohort[observed_target] >= TARGET_GOALS).astype(int)
train = cohort.loc[cohort["season_start_year"] < TEST_YEAR]
test = cohort.loc[cohort["season_start_year"] == TEST_YEAR]
if train.empty or test.empty or train["target"].nunique() < 2:
    raise ValueError("Choose a test year/cohort with training and test records and both training classes.")
print(f"Training: {len(train):,}; test: {len(test):,}")
print(test["target"].value_counts())


Training: 3,882; test: 348
target
0    184
1    164
Name: count, dtype: int64


## Compare classifiers
Use an explicit feature list. No future outcome or reference-table current valuation is a predictor.
Preprocessing is fitted on training data only. Repeated players across years are intentional here;
use player-disjoint evaluation for a question about previously unseen players.
Profile positions/heights are snapshot fields, not historical observations.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix

numeric = ["age", "height_cm", "goals_per_season", "assists_per_season", "minutes_played",
           "appearances", "goals_per90", "market_value_before_first_match_eur"]
categorical = ["league", "sub_position", "foot"]
features = numeric + categorical
models = {
    "baseline": DummyClassifier(strategy="most_frequent"),
    "logistic_regression": LogisticRegression(max_iter=2000, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
}
results, fitted_models = [], {}
for name, model in models.items():
    prepare = ColumnTransformer([
        ("numbers", Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), numeric),
        ("categories", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                 ("encode", OneHotEncoder(handle_unknown="ignore"))]), categorical),
    ])
    pipeline = Pipeline([("prepare", prepare), ("model", model)])
    pipeline.fit(train[features], train["target"])
    predicted = pipeline.predict(test[features])
    fitted_models[name] = pipeline
    results.append({"model": name, "accuracy": accuracy_score(test["target"], predicted),
                    "recall_macro": recall_score(test["target"], predicted, average="macro", zero_division=0),
                    "f1_macro": f1_score(test["target"], predicted, average="macro", zero_division=0)})
    print(name, "confusion matrix (rows actual, columns predicted; labels 0, 1):")
    print(confusion_matrix(test["target"], predicted, labels=[0, 1]))
pd.DataFrame(results)


baseline confusion matrix (rows actual, columns predicted; labels 0, 1):
[[184   0]
 [164   0]]
logistic_regression confusion matrix (rows actual, columns predicted; labels 0, 1):
[[152  32]
 [ 70  94]]
random_forest confusion matrix (rows actual, columns predicted; labels 0, 1):
[[147  37]
 [ 65  99]]


,model,accuracy,recall_macro,f1_macro
0,baseline,0.528736,0.500000,0.345865
1,logistic_regression,0.706897,0.699629,0.698522
2,random_forest,0.706897,0.701286,0.701212


## Next experiments
Change the league/position filters, compare held-out leagues, or use the existing model registry
for more classifiers. For match prediction, load `player_match` and start with `prior_*` features;
current-match goals/minutes and final scores reveal the outcome. Consult
`docs/transfermarkt_tables.md` and `data_dictionary.csv` before choosing new features.
